# GATE froth (FO1) — extracción C2 (DINOv3) y C3 (V-JEPA 2.1) sobre IEEE

**Qué hace**: extrae los embeddings de las condiciones C2 y C3 del GATE froth
(`local_docs/froth/froth.md` §8) sobre las 2.386 secuencias del dataset IEEE
"Flotation Froth Sequence Images", incluido el **control de permutación temporal** de C3.
C1 (geométrico industrial) se computa LOCAL con `froth_gate/extract_c1.py` — no acá.

**Prerequisito CUMPLIDO (verificado 2026-07-06)**: los 4 zips ya están en Drive en
`Amta_lab/datasets/Dataset: "Flotation Froth Sequence Images"/` (subidos por Daniel López);
la celda 3 los encuentra por glob. El checkpoint V-JEPA ViT-B también está en
`Amta_lab/models/` (no re-descarga).

**Runtime**: T4 estándar (política del repo). Tiempo estimado: ~35–50 min todo.

**Salidas** (VM → persistidas a `Amta_lab/outputs/gate_froth/`):
`c2_dinov3_vits.npz`, `c2_dinov3_vitb.npz`, `c3_vjepa21_vitb.npz`,
`c3_vjepa21_vitb_PERM.npz`, `costs.json`.
Al terminar: bajarlas a `froth_gate/results/` local y correr `python gate_analysis.py`.

**Al terminar la sesión: Remove Server** (política de unidades).

In [ ]:
# 1) Deps + GPU
!pip install -q --upgrade "transformers>=4.56.0" huggingface_hub timm einops
import torch, time, os, sys, io, re, json, zipfile, shutil
import numpy as np
from PIL import Image

assert torch.cuda.is_available(), "sin GPU: elegir runtime T4"
print(torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
DEVICE = "cuda"
# bf16 solo nativo (sm80+); en T4 SIEMPRE fp16 (gotcha 2b del smoke test)
bf16_nativo = torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if bf16_nativo else torch.float16
print("autocast dtype:", DTYPE)

In [ ]:
# 2) Drive + Amta_lab (patrón del smoke test)
from google.colab import drive
drive.mount("/content/drive")

AMTA_LAB = "/content/drive/MyDrive/Amta_lab"
for _sub in ("models", "data", "outputs"):
    os.makedirs(os.path.join(AMTA_LAB, _sub), exist_ok=True)
HUB_CKPTS_VM = os.path.expanduser("~/.cache/torch/hub/checkpoints")

def sync_ckpt_drive(nombre):
    en_drive = os.path.join(AMTA_LAB, "models", nombre)
    en_vm = os.path.join(HUB_CKPTS_VM, nombre)
    os.makedirs(HUB_CKPTS_VM, exist_ok=True)
    if os.path.exists(en_drive) and not os.path.exists(en_vm):
        print(f"checkpoint Drive -> VM: {nombre}"); shutil.copy(en_drive, en_vm)
    elif os.path.exists(en_vm) and not os.path.exists(en_drive):
        print(f"checkpoint VM -> Drive: {nombre}"); shutil.copy(en_vm, en_drive)

OUT_DIR = "/content/gate_froth"; os.makedirs(OUT_DIR, exist_ok=True)
print("Amta_lab listo")

In [ ]:
# 3) Datos: zips Drive -> disco de la VM (I/O rápido; la VM es el filesystem de trabajo)
# Los 4 zips YA están en Drive (subidos por Daniel López 2026-07-03, verificado 2026-07-06):
# Amta_lab/datasets/Dataset: "Flotation Froth Sequence Images"/class Ⅰ..Ⅳ.zip
import glob as _g
cands = (_g.glob(os.path.join(AMTA_LAB, "datasets", "*Flotation Froth Sequence*"))
         + _g.glob(os.path.join(AMTA_LAB, "data", "ieee_froth")))
assert cands, f"no encuentro la carpeta de zips IEEE bajo {AMTA_LAB}/datasets ni /data"
DATA_DRIVE = cands[0]
print("origen Drive:", DATA_DRIVE)

DATA_VM = "/content/ieee_froth"
os.makedirs(DATA_VM, exist_ok=True)
zips = sorted(f for f in os.listdir(DATA_DRIVE) if f.endswith(".zip"))
assert len(zips) == 4, f"esperaba 4 zips en {DATA_DRIVE}, hay: {zips}"
for f in zips:
    dst = os.path.join(DATA_VM, f)
    if not os.path.exists(dst):
        print("copiando", f); shutil.copy(os.path.join(DATA_DRIVE, f), dst)
print("zips en VM:", os.listdir(DATA_VM))

In [ ]:
# 4) Índice de secuencias — MISMO orden y keys que extract_c1.py local (join por clase+seq_id)
SECUENCIAS = []  # (zip_path, seq_id:str, clase:int)
for k, zname in enumerate(sorted(os.listdir(DATA_VM)), 1):
    zp = os.path.join(DATA_VM, zname)
    with zipfile.ZipFile(zp) as z:
        seqs = sorted({m.group(1) for n in z.namelist() if (m := re.search(r"/(\d+)/\d+\.jpg$", n))})
    SECUENCIAS += [(zp, s, k) for s in seqs]
    print(f"clase {k}: {zname} -> {len(seqs)} secuencias")
print("total:", len(SECUENCIAS))

_ZCACHE = {}
def abrir_zip(zp):
    if zp not in _ZCACHE: _ZCACHE[zp] = zipfile.ZipFile(zp)
    return _ZCACHE[zp]

def cargar_frames(zp, seq_id):
    """12 frames PIL grayscale, orden 1001..1012 (lexicográfico = temporal)."""
    z = abrir_zip(zp)
    names = sorted(n for n in z.namelist() if re.search(rf"/{seq_id}/\d+\.jpg$", n))
    return [Image.open(io.BytesIO(z.read(n))).convert("L") for n in names]

# misma receta de semilla de permutación que extract_c1.py (control §8.4)
def perm_de(seq_id, clase, n=12):
    seed = 42_000 + clase * 10_000 + int(seq_id)
    return np.random.RandomState(seed).permutation(n)

COSTS = {}

## C2 — DINOv3 frozen (ViT-S/16 y ViT-B/16, dos puntos de la curva de Pareto)

Por frame: token CLS + mean(patch tokens) concatenados → por secuencia: **mean+std sobre
los 12 frames** (pooling declarado en froth.md §8.2; invariante a permutación temporal
por construcción — por eso C2 no necesita pasada PERM).
DINOv3 es *gated* en HF: la celda pide login (mismo acceso ya usado en `dino_v3/`).

In [ ]:
# 5) C2: autenticación HF (DINOv3 es gated: hay que aceptar la licencia en el hub)
#
# Cascada de fuentes del token, pensada para correr desde VS Code (donde el vault de
# secrets de Colab NO es accesible: "Secrets can only be fetched when running from the
# Colab UI") tanto como desde la UI web de Colab:
#   1. variable de entorno HF_TOKEN
#   2. token ya persistido en la VM (~/.cache/huggingface)
#   3. Amta_lab/.hf_token en Drive  <- sobrevive a la VM; se escribe una sola vez
#   4. secret HF_TOKEN del vault de Colab (solo si se corre desde la UI web)
#   5. getpass interactivo (último recurso; ofrece guardar en Drive para la próxima VM)
GUARDAR_TOKEN_EN_DRIVE = True   # escribe el token en tu Drive PRIVADO (Amta_lab/.hf_token)

import os
from huggingface_hub import login, whoami

TOKEN_DRIVE = os.path.join(AMTA_LAB, ".hf_token")

def _obtener_token():
    tok = os.environ.get("HF_TOKEN", "").strip()
    if tok:
        print("token: variable de entorno HF_TOKEN"); return tok, False
    try:
        whoami(); print("token: ya autenticado en esta VM"); return None, False
    except Exception:
        pass
    if os.path.exists(TOKEN_DRIVE):
        tok = open(TOKEN_DRIVE).read().strip()
        if tok:
            print("token: Amta_lab/.hf_token (Drive)"); return tok, False
    try:  # solo funciona desde la UI web de Colab
        from google.colab import userdata
        tok = (userdata.get("HF_TOKEN") or "").strip()
        if tok:
            print("token: secret HF_TOKEN del vault de Colab"); return tok, True
    except Exception as e:
        print(f"vault de Colab no disponible ({type(e).__name__}) -> sigo con la cascada")
    from getpass import getpass
    tok = getpass("HF token (hf_...; https://huggingface.co/settings/tokens): ").strip()
    return tok, True

_tok, _es_nuevo = _obtener_token()
if _tok:
    login(token=_tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = _tok
    if _es_nuevo and GUARDAR_TOKEN_EN_DRIVE:
        with open(TOKEN_DRIVE, "w") as fh:
            fh.write(_tok)
        try:
            os.chmod(TOKEN_DRIVE, 0o600)
        except OSError:
            pass   # el FUSE de Drive ignora los permisos POSIX
        print(f"token guardado en {TOKEN_DRIVE} (Drive privado) para las próximas VMs")

print("HF autenticado como:", whoami()["name"])
del _tok   # no dejar el token colgando en el namespace del notebook

# Chequeo temprano de acceso a los repos gated: mejor fallar acá que a mitad de la extracción.
from huggingface_hub import model_info
for _repo in ("facebook/dinov3-vits16-pretrain-lvd1689m",
              "facebook/dinov3-vitb16-pretrain-lvd1689m"):
    try:
        model_info(_repo); print("acceso OK:", _repo)
    except Exception as e:
        raise RuntimeError(
            f"sin acceso a {_repo}: {type(e).__name__}. DINOv3 es gated — aceptá la "
            f"licencia en https://huggingface.co/{_repo} con la MISMA cuenta del token."
        ) from e


In [ ]:
# 6) C2: extracción (S y B)
from transformers import AutoImageProcessor, AutoModel

def extraer_c2(model_name, tag, batch_size=48):
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    torch.cuda.reset_peak_memory_stats()
    embs, keys, t_gpu = [], [], 0.0
    buf_img, buf_key = [], []

    def flush():
        nonlocal t_gpu
        if not buf_img: return
        inputs = processor(images=[im.convert("RGB") for im in buf_img], return_tensors="pt").to(DEVICE)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.inference_mode(), torch.autocast("cuda", dtype=DTYPE):
            out = model(**inputs).last_hidden_state  # (B, 1+reg+P, D)
        torch.cuda.synchronize(); t_gpu += time.perf_counter() - t0
        nreg = getattr(model.config, "num_register_tokens", 0)
        cls = out[:, 0]; patches = out[:, 1 + nreg:].mean(1)
        embs.append(torch.cat([cls, patches], -1).float().cpu().numpy())
        keys.extend(buf_key); buf_img.clear(); buf_key.clear()

    t_all = time.time()
    for zp, seq_id, clase in SECUENCIAS:
        for j, im in enumerate(cargar_frames(zp, seq_id)):
            buf_img.append(im); buf_key.append((clase, seq_id, j))
            if len(buf_img) >= batch_size: flush()
    flush()
    F = np.concatenate(embs)                      # (n_frames_total, 2D)
    # agregación por secuencia: mean+std sobre los 12 frames
    porseq, clases, seqids = [], [], []
    idx = {}
    for i, (c, s, j) in enumerate(keys): idx.setdefault((c, s), []).append(i)
    for (c, s), ii in idx.items():
        v = F[ii]; porseq.append(np.concatenate([v.mean(0), v.std(0)]))
        clases.append(c); seqids.append(s)
    out_npz = os.path.join(OUT_DIR, f"c2_dinov3_{tag}.npz")
    np.savez_compressed(out_npz, emb=np.array(porseq, dtype=np.float32),
                        clase=np.array(clases), seq_id=np.array(seqids))
    COSTS[f"c2_{tag}"] = {"model": model_name, "params_M": n_params / 1e6,
                          "t_gpu_total_s": t_gpu, "ms_por_frame": t_gpu / len(keys) * 1e3,
                          "t_total_s": time.time() - t_all, "input_res": 224,
                          "vram_peak_GB": torch.cuda.max_memory_allocated() / 1e9,
                          "dim_por_seq": len(porseq[0])}
    print(tag, COSTS[f"c2_{tag}"])
    del model; torch.cuda.empty_cache()

extraer_c2("facebook/dinov3-vits16-pretrain-lvd1689m", "vits")
extraer_c2("facebook/dinov3-vitb16-pretrain-lvd1689m", "vitb")

## C3 — V-JEPA 2.1 ViT-B 384 frozen (+ control de permutación temporal)

Patrón validado en `v-jepa-2/colab/vjepa2_smoke_test.ipynb`: hub `pretrained=False` +
checkpoint manual de `dl.fbaipublicfiles.com` (`ema_encoder`, strict=True) + **parche RoPE**
(fp32 interno / salida en dtype de entrada) + **fp16 en T4**. Clip = 12 frames → tubelet 2
→ 6×(384/16)² = 3456 tokens. Pooling: mean+std sobre tokens → 1536 dims.
La pasada PERM re-encodea cada clip con los frames en orden permutado (misma semilla que
`extract_c1.py`): si el F1 de C3 no cae con PERM, la señal no es temporal (froth.md §8.4).

In [ ]:
# 7) C3: arquitectura + checkpoint + parche
VJEPA_BASE_URL = "https://dl.fbaipublicfiles.com/vjepa2"
CKPT_FILE, CKPT_KEY, CKPT_STRICT = "vjepa2_1_vitb_dist_vitG_384.pt", "ema_encoder", True

encoder, predictor = torch.hub.load("facebookresearch/vjepa2", "vjepa2_1_vit_base_384",
                                    pretrained=False, trust_repo=True)
del predictor
sync_ckpt_drive(CKPT_FILE)
state = torch.hub.load_state_dict_from_url(f"{VJEPA_BASE_URL}/{CKPT_FILE}", map_location="cpu")
sync_ckpt_drive(CKPT_FILE)
sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in state[CKPT_KEY].items()}
print("load_state_dict:", encoder.load_state_dict(sd, strict=CKPT_STRICT))
del state, sd
encoder = encoder.cuda().eval()
for p in encoder.parameters(): p.requires_grad_(False)
torch.cuda.empty_cache()
N_PARAMS_C3 = sum(p.numel() for p in encoder.parameters())
print(f"encoder: {N_PARAMS_C3/1e6:.0f}M params | embed_dim {encoder.embed_dim}")

# parche RoPE (gotcha 2 del smoke test; idempotente)
import glob as _glob
hub_repo = _glob.glob(os.path.join(torch.hub.get_dir(), "facebookresearch_vjepa2_*"))[0]
sys.path.insert(0, hub_repo)
import app.vjepa_2_1.models.utils.modules as m21
if not getattr(m21.rotate_queries_or_keys, "_amta_patched", False):
    _rope_orig = m21.rotate_queries_or_keys
    def _rope_fp32(x, pos, n_registers, has_cls_first):
        return _rope_orig(x.float(), pos, n_registers=n_registers, has_cls_first=has_cls_first).to(x.dtype)
    _rope_fp32._amta_patched = True
    m21.rotate_queries_or_keys = _rope_fp32
    print("parche RoPE aplicado")

In [ ]:
# 8) C3: extracción normal + PERM
import torchvision.transforms.v2.functional as TF

IMG_SIZE, T_FRAMES = 384, 12
SHORT_SIDE = int(256 / 224 * IMG_SIZE)  # 438, protocolo eval estándar
IM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1)
IM_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1)

def clip_de(frames_pil, order=None):
    if order is not None: frames_pil = [frames_pil[i] for i in order]
    t = torch.stack([torch.from_numpy(np.asarray(f)) for f in frames_pil])   # (T,H,W) uint8
    t = t.unsqueeze(1).repeat(1, 3, 1, 1).float() / 255.0                     # (T,3,H,W)
    t = TF.center_crop(TF.resize(t, SHORT_SIDE, antialias=True), IMG_SIZE)    # (T,3,384,384)
    t = t.permute(1, 0, 2, 3)                                                 # (3,T,H,W)
    return (t - IM_MEAN) / IM_STD

def extraer_c3(permutado, tag, batch_size=8):
    torch.cuda.reset_peak_memory_stats()
    embs, clases, seqids, t_gpu = [], [], [], 0.0
    buf = []
    def _forward(sub):
        nonlocal t_gpu
        clips = torch.stack([b[0] for b in sub]).cuda()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.inference_mode(), torch.autocast("cuda", dtype=DTYPE):
            f = encoder(clips)                    # (B, 3456, D)
            e = torch.cat([f.mean(1), f.std(1)], -1)
        torch.cuda.synchronize(); t_gpu += time.perf_counter() - t0
        embs.append(e.float().cpu().numpy())
        clases.extend(b[1] for b in sub); seqids.extend(b[2] for b in sub)

    def flush():
        if not buf: return
        try:
            _forward(buf)
        except torch.cuda.OutOfMemoryError:
            # red de seguridad: 40 min de extraccion no pueden morir por un pico de VRAM.
            # Se parte el batch a la mitad; el resultado es identico (no hay estado entre clips).
            torch.cuda.empty_cache()
            mitad = max(1, len(buf) // 2)
            print(f"OOM con batch {len(buf)} -> reintento en trozos de {mitad}", flush=True)
            for i in range(0, len(buf), mitad):
                _forward(buf[i:i + mitad]); torch.cuda.empty_cache()
        buf.clear()
    t_all = time.time()
    for i, (zp, seq_id, clase) in enumerate(SECUENCIAS):
        frames = cargar_frames(zp, seq_id)
        order = perm_de(seq_id, clase, len(frames)) if permutado else None
        buf.append((clip_de(frames, order), clase, seq_id))
        if len(buf) >= batch_size: flush()
        if (i + 1) % 400 == 0: print(f"{i+1}/{len(SECUENCIAS)} ({time.time()-t_all:.0f}s)")
    flush()
    np.savez_compressed(os.path.join(OUT_DIR, f"c3_vjepa21_vitb{tag}.npz"),
                        emb=np.concatenate(embs).astype(np.float32),
                        clase=np.array(clases), seq_id=np.array(seqids))
    COSTS[f"c3{tag}"] = {"model": "vjepa2_1_vit_base_384", "params_M": N_PARAMS_C3 / 1e6,
                         "t_gpu_total_s": t_gpu, "ms_por_clip": t_gpu / len(SECUENCIAS) * 1e3,
                         "t_total_s": time.time() - t_all, "input": f"{T_FRAMES}x{IMG_SIZE}",
                         "vram_peak_GB": torch.cuda.max_memory_allocated() / 1e9,
                         "dim_por_seq": int(embs[0].shape[1])}
    print(f"c3{tag}", COSTS[f"c3{tag}"])

extraer_c3(False, "")        # C3 normal
extraer_c3(True, "_PERM")    # control de permutación temporal (§8.4)

In [ ]:
# 9) Persistir a Drive (Amta_lab/outputs/gate_froth) + costos
with open(os.path.join(OUT_DIR, "costs.json"), "w") as f:
    json.dump(COSTS, f, indent=2)
dst = os.path.join(AMTA_LAB, "outputs", "gate_froth")
os.makedirs(dst, exist_ok=True)
for f in os.listdir(OUT_DIR):
    shutil.copy(os.path.join(OUT_DIR, f), os.path.join(dst, f))
    print("->", os.path.join(dst, f))
print(json.dumps(COSTS, indent=2))

## Siguiente paso (local)

1. Bajar los `.npz` + `costs.json` de `Amta_lab/outputs/gate_froth/` a
   `froth_gate/results/` en la máquina local (web de Drive o extensión Colab).
2. Correr `python froth_gate/gate_analysis.py` → emite la tabla C1/C2/C3, el control de
   permutación y el veredicto **GO / NO-GO** con los umbrales congelados de froth.md §8.3.
3. **Remove Server.**

In [ ]:
import sys, os
print("python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
except Exception as e:
    print("torch:", type(e).__name__, e)
print("SECUENCIAS:", len(globals().get("SECUENCIAS", [])))
print("encoder cargado:", "encoder" in globals())
print("OUT_DIR:", globals().get("OUT_DIR"))
print("AMTA_LAB montado:", os.path.isdir(globals().get("AMTA_LAB", "/nope")))
print("DTYPE:", globals().get("DTYPE"))